# M10 — Shared movies: ISC, temporal encoding, and alignment

**Original guided lab · 75–100 minutes.** Read 20 min, predict/code 35 min, failure investigation 20 min, explain and transfer 15 min. Run all cells in order in a fresh kernel. All executed data are synthetic unless explicitly stated. No network, GPU, or external dataset is required.

Naturalistic experiments present rich continuous stimuli such as stories and films. Their timing creates structure shared across participants, but also strong temporal dependence. Several analyses ask different questions about these data. Intersubject correlation asks whether responses at corresponding locations co-fluctuate across people. Encoding relates stimulus features to responses. Functional alignment seeks correspondences among response patterns that anatomical registration may not fully align.

A leave-one-out ISC compares one person with an average formed from the other people. Leaving the person out matters because including their own noise in the reference inflates similarity. Fisher transformation is often used to summarize correlations, but it does not make the underlying subject comparisons independent. Each leave-one-out estimate shares data with other estimates. Inference must respect this dependence and the temporal structure; an ordinary independent t test over every pair is not automatically appropriate.

Our synthetic movie response combines a shared smooth signal and participant noise. We inspect ISC with the correct clock, then shift one person's time series. The row count and variance stay the same, while synchrony drops. A real mismatch might arise from stimulus onset, discarded volumes, resampling, or missing segments. Do not fix it by trying every alignment and reporting the best test correlation without an explicit estimation and evaluation plan.

Encoding requires feature timing that corresponds to the retained scans and an appropriate hemodynamic model. A train/test split across a continuous time series needs independent blocks, runs, stimuli, or a justified buffer; random time samples can exploit autocorrelation. We demonstrate train-only ridge fitting with a gap between blocks. This is a simplified timing exercise, not a realistic hemodynamic encoding model. The generator directly places known features into the response, and that assumption is stated rather than hidden.

Functional alignment is distinct from scanner-space registration. A shared response model estimates lower-dimensional common responses and participant-specific mappings. Training the mappings and evaluating on the same time segment can overstate success. The upstream BrainIAK SRM and Dartmouth functional-alignment tutorials explicitly provide a next assignment, while this core lab limits execution to ISC and temporal encoding.

Ask AI to name the axis and identity of every average: across participants, time, voxels, or stimulus features. Those operations are not interchangeable. A high ISC establishes shared response structure under the specified comparison, not a complete explanation of the represented content or a universal measure of engagement. Shared artifacts can also synchronize. Preserve stimulus clocks, participant IDs, preprocessing decisions, and the independence unit for each downstream statistic.

## Transformation contract

Aligned time×participant responses → leave-one-out references → one ISC per participant. Feature time series → training-only encoding model → predictions in a separated test block. Clock shifts preserve samples but break temporal correspondence.

## Ask your AI tutor

```text
Explain this notebook one transformation at a time.
Before each cell ask me to predict shapes, units, and a check.
Give edits in executable cells of at most 20 lines.
Keep the prescribed split, random seed, and tests intact.
Distinguish generated suggestions from executed results.
After the failure experiment, ask me to explain the mechanism.
```

In [1]:
import numpy as np
from scipy.ndimage import gaussian_filter1d
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
rng=np.random.default_rng(410)
shared=gaussian_filter1d(rng.normal(size=500),3);shared/=shared.std()
Y=shared[:,None]+rng.normal(0,.6,(500,8))
def loo_isc(data):
    return np.array([np.corrcoef(data[:,i],np.delete(data,i,axis=1).mean(axis=1))[0,1] for i in range(data.shape[1])])
isc=loo_isc(Y);shifted=Y.copy();shifted[:,0]=np.roll(shifted[:,0],90)
print('Mean correct ISC / shifted participant ISC:',isc.mean(),loo_isc(shifted)[0])
assert isc.mean()>.75 and abs(loo_isc(shifted)[0])<.3


Mean correct ISC / shifted participant ISC: 0.8377253039718606 0.21862448253028466


In [2]:
features=np.column_stack([shared,gaussian_filter1d(rng.normal(size=500),2)])
response=features@np.array([2.,-1.])+rng.normal(0,.3,500)
train=np.arange(0,280);test=np.arange(320,500)
model=Ridge(alpha=1).fit(features[train],response[train])
score=r2_score(response[test],model.predict(features[test]))
print('Separated-block encoding R²:',score,'unused gap:',test.min()-train.max()-1)
assert score>.9 and train.max()<test.min()-20
self_reference=np.corrcoef(Y[:,0],Y.mean(axis=1))[0,1]
print('Self-inclusive / leave-one-out reference:',self_reference,isc[0])
assert self_reference>isc[0]


Separated-block encoding R²: 0.9694900328296571 unused gap: 40
Self-inclusive / leave-one-out reference: 0.8870261084462621 0.8492100419648768


## Deliberate failure and repair

Including a participant in their own reference inflates ISC. Shifting the clock can destroy true synchrony while passing ordinary array checks. Repair the averaging and timing contracts, then evaluate the repair on data not used to select it. Randomly shuffling individual time points is not a general null procedure for autocorrelated responses.

## Your investigation

Inspect the vendored ISC notebook and identify its stimulus, participant count, averaging rule, and inference unit before running external data. Compare phase randomization, circular shift, and subject bootstrap conceptually; each preserves and breaks different structure. Plan a shared-response-model test using separate training and evaluation movie segments.

## Transfer to real neuroimaging

The upstream ISC lab uses naturalistic fMRI with actual masks and stimulus timing. The SRM extension requires BrainIAK and its data setup. No real movie data, SRM fitting, or ISC significance test is executed in the offline core.

**Primary teaching sources, pinned where hosted on GitHub:**

- [BrainIAK: ISC and ISFC (vendored)](https://github.com/brainiak/brainiak-tutorials/blob/fb62ede943d9694fe703aee0df5f43ecf5558415/tutorials/10-isc.ipynb)
- [BrainIAK: shared response modeling](https://github.com/brainiak/brainiak-tutorials/blob/fb62ede943d9694fe703aee0df5f43ecf5558415/tutorials/11-srm.ipynb)
- [Dartmouth/OHBM: functional alignment](https://github.com/naturalistic-data-analysis/naturalistic_data_analysis/blob/88bd22741508b4d678202bde7530cee0511e283f/content/Functional_Alignment.ipynb)

Pinned upstream tutorials are a separate assignment; they have **not been executed** by this core lab. They may require data downloads, specialist dependencies, unfinished student cells, and additional compute.

## Exit questions and answer key

1. Why exclude the person from their ISC reference? **To avoid correlating their measurement noise with itself.**
2. Does random time-point splitting establish new-stimulus generalization? **No; neighboring samples and shared stimulus content can leak information.**